# Zero Trust Authentication Based on ROS2
本專題以公開的 [ROSpace: Intrusion Detection Dataset for a ROS2-Based Cyber-Physical System](https://arxiv.org/abs/2402.08468) 為基礎，並參考其作者提供的程式碼庫 [rospace_dataset](https://github.com/TommasoPuccetti/rospace_dataset) 。我們的目標是復刻並簡化原始流程，建立一個可操作的版本，透過資料集訓練異常檢測模型，進一步驗證 Zero Trust 架構在 ROS2 環境下的可行性。

> This project is based on the publicly available [ROSpace: Intrusion Detection Dataset for a ROS2-Based Cyber-Physical System](https://arxiv.org/abs/2402.08468) and references the code repository [rospace_dataset](https://github.com/TommasoPuccetti/rospace_dataset) provided by its authors. Our goal is to replicate and simplify the original process, build a workable version, train an anomaly detection model using the dataset, and further verify the feasibility of the Zero Trust architecture in a ROS2 environment.

整個流程我們將其拆分成 Data Collection、Data Preprcessing、Model Training、Application 做簡單的區分。

### 1. Data Collection
架構可以參考 [Simple Architecture Image](README/architecture.png)，主要的攻擊流程：

##### (1) NMAP Discovery / Port Scanning

##### (2) ROS2 Reconnaissance

##### (3) NMAP SYN Flood (IPv6 RA Flood)

##### (4) ROS2 Reflection

##### (5) ROS2 Node Crashing

##### (6) Metasplot SYN Flood


### 2. Data Preprocessing

In [2]:
import os
import subprocess
import pandas as pd

print("< Path Setting >")
date_val = input("input code of collected data (e.g. 0507, 0509): ")

# get root path
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"
if project_name in current_dir:
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

# path combinations
processing_dir = os.path.join(root_dir, "rospace_dataset", "1_processing")
labeling_dir = os.path.join(root_dir, "rospace_dataset", "2_labeling")

pcapng_path = os.path.join(root_dir, "raspberry_subscriber", f"monitor_{date_val}", "Tshark", "temp_capture.pcapng")
os_path = os.path.join(root_dir, "raspberry_subscriber", f"monitor_{date_val}", "OS", "OS_monitor.csv")
ros_path = os.path.join(root_dir, "raspberry_subscriber", f"monitor_{date_val}", "ROSBags", "ros2_monitor.csv")
attack_path = os.path.join(root_dir, "rospace_dataset", "attack", f"attacks_log_{date_val}.csv")

# if not os.path.exists(pcapng_path):
#     raise FileNotFoundError(f"找不到封包檔案，請確認日期是否輸入正確或檔案是否存在：\n{pcapng_path}")

print("Done.")

< Path Setting >
Done.


In [3]:
# import subprocess

print("< PCAPNG to CSV Conversion >")
try:
    result = subprocess.run(
        ["python", "custom_pcapng2csv.py", pcapng_path],
        cwd=processing_dir,
        check=True,
        capture_output=True
    )
    print("Done.")
    
except subprocess.CalledProcessError as e:
    print("Exception Code:", e.returncode)
    print("Error Input:", e.cmd)
    print("Output Message:", e.output)
    print("Error Message:", e.stderr)

< PCAPNG to CSV Conversion >
Done.


In [4]:
converted_net_path = os.path.join(processing_dir, "converted", "temp_capture.csv")
df = pd.read_csv(converted_net_path, low_memory=False)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# print("Data:")
# print(df.head())

print("Info:")
print(df.info())
# print("Describe:")
# print(df.describe())

Rows: 24407
Columns: ['Unnamed: 0', 'layers.frame.frame.section_number', 'layers.frame.frame.interface_id', 'layers.frame.frame.interface_id_tree.frame.interface_name', 'layers.frame.frame.encap_type', 'layers.frame.frame.time', 'layers.frame.frame.time_utc', 'layers.frame.frame.time_epoch', 'layers.frame.frame.offset_shift', 'layers.frame.frame.time_relative', 'layers.frame.frame.number', 'layers.frame.frame.len', 'layers.frame.frame.cap_len', 'layers.frame.frame.marked', 'layers.frame.frame.ignored', 'layers.frame.frame.protocols', 'layers.frame.frame.encoding', 'layers.eth.eth.dst', 'layers.eth.eth.dst_tree.eth.dst_resolved', 'layers.eth.eth.dst_tree.eth.dst.oui', 'layers.eth.eth.dst_tree.eth.dst.oui_resolved', 'layers.eth.eth.dst_tree.eth.dst.lg', 'layers.eth.eth.dst_tree.eth.dst.ig', 'layers.eth.eth.dst_tree.eth.addr', 'layers.eth.eth.dst_tree.eth.addr_resolved', 'layers.eth.eth.dst_tree.eth.addr.oui', 'layers.eth.eth.dst_tree.eth.addr.oui_resolved', 'layers.eth.eth.dst_tree.eth.l

In [5]:
# import subprocess

print("< Dataset Merging >")
try:
    subprocess.run(
        ["python", "csv_merging_parallel.py", "-s", os_path, "-n", converted_net_path, "-r", ros_path, "-a", attack_path],
        cwd=processing_dir,
        check=True
    )
    
    print("Done. Please check the csv_merging_parallel.py for the output.")

except subprocess.CalledProcessError:
    print("Please check the subprocess.")


< Dataset Merging >
Done. Please check the csv_merging_parallel.py for the output.


In [10]:
converted_net_path = os.path.join(processing_dir, "merged-dataset", f"merged-{date_val}_unlabelled.csv")
df = pd.read_csv(converted_net_path)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# print("Data:")
# print(df.head())

print("Info:")
print(df.info())
print("Describe:")
print(df.describe())

C:\Users\yuyux\AppData\Local\Temp\ipykernel_22004\3377972160.py:2: DtypeWarning: Columns (0: layers.rtps.rtps.magic, 1: layers.rtps.rtps.version, 2: layers.rtps.rtps.vendorId, 3: layers.rtps.rtps.guidPrefix.src, 4: layers.rtps.rtps.guidPrefix.src_tree.rtps.hostId, 5: layers.rtps.rtps.guidPrefix.src_tree.rtps.appId, 6: layers.rtps.rtps.guidPrefix.src_tree.rtps.sm.guidPrefix.instanceId, 7: layers.rtps.rtps.guidPrefix, 8: layers.rtps.rtps.sm.id, 9: layers.rtps.rtps.sm.id_tree.rtps.sm.flags, 10: layers.rtps.rtps.sm.id_tree.rtps.extra_flags, 11: layers.rtps.rtps.sm.id_tree.rtps.sm.rdEntityId, 12: layers.rtps.rtps.sm.id_tree.rtps.sm.rdEntityId_tree.rtps.sm.rdEntityId.entityKey, 13: layers.rtps.rtps.sm.id_tree.rtps.sm.rdEntityId_tree.rtps.sm.rdEntityId.entityKind, 14: layers.rtps.rtps.sm.id_tree.rtps.sm.wrEntityId, 15: layers.rtps.rtps.sm.id_tree.rtps.sm.wrEntityId_tree.rtps.sm.wrEntityId.entityKey, 16: layers.rtps.rtps.sm.id_tree.rtps.sm.wrEntityId_tree.rtps.sm.wrEntityId.entityKind, 17: lay

Rows: 23233
Columns: ['Unnamed: 0', 'timestamp', 'layers.frame.frame.interface_id', 'layers.frame.frame.interface_id_tree.frame.interface_name', 'layers.frame.frame.encap_type', 'layers.frame.frame.time', 'layers.frame.frame.time_utc', 'layers.frame.frame.offset_shift', 'layers.frame.frame.time_relative', 'layers.frame.frame.number', 'layers.frame.frame.len', 'layers.frame.frame.cap_len', 'layers.frame.frame.marked', 'layers.frame.frame.ignored', 'layers.frame.frame.protocols', 'layers.frame.frame.encoding', 'layers.eth.eth.dst', 'layers.eth.eth.dst_tree.eth.dst_resolved', 'layers.eth.eth.dst_tree.eth.dst.oui', 'layers.eth.eth.dst_tree.eth.dst.oui_resolved', 'layers.eth.eth.dst_tree.eth.dst.lg', 'layers.eth.eth.dst_tree.eth.dst.ig', 'layers.eth.eth.dst_tree.eth.addr', 'layers.eth.eth.dst_tree.eth.addr_resolved', 'layers.eth.eth.dst_tree.eth.addr.oui', 'layers.eth.eth.dst_tree.eth.addr.oui_resolved', 'layers.eth.eth.dst_tree.eth.lg', 'layers.eth.eth.dst_tree.eth.ig', 'layers.eth.eth.src

In [11]:
# import subprocess

# Prints only the Pandas version
print("< Dataset Labeling >")

try:
    subprocess.run(
        ["python", "labeling.py", "-s", converted_net_path, "-a", attack_path],
        cwd=labeling_dir,
        check=True
    )
    print("Done.")
    
except subprocess.CalledProcessError as e:
    print("Exception Code:", e.returncode)
    print("Error Input:", e.cmd)
    print("Output Message:", e.output)
    print("Error Message:", e.stderr)

< Dataset Labeling >
Done.


### 3. Model Training

### 4. Application